### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some reasons in favor of choosing AutoGen for your new AI Agent project:

1. **Efficiency in Multi-Agent Collaboration**: AutoGen facilitates seamless collaboration between multiple AI agents, which can lead to enhanced performance and functionality in complex applications.

2. **Ease of Use**: The framework simplifies the development process for LLM-powered applications, allowing developers to focus on building their applications rather than dealing with intricate agent coordination protocols.

3. **Robust Scalability**: AutoGen is designed to support scalable multi-agent systems, making it suitable for projects that may require expansion or adaptation over time.

4. **Integration with Secure Code Executors**: It can integrate with secure code execution environments like Docker, ensuring that agents can run code safely while managing potential security risks.

5. **Backed by Microsoft**: Being developed and supported by Microsoft provides confidence in its stability, regular updates, and a wealth of practical usage scenarios.

6. **Open Source**: The open-source nature of AutoGen allows developers to modify and enhance the framework according to their specific needs without any revenue pressure.

7. **Shortened Feedback Loops**: The design enables other AI agents to give feedback to one another, which can accelerate the learning process and reduce the workload for developers.

These benefits make AutoGen a compelling choice for building and scaling AI agent applications effectively. 

TERMINATE

## Cons of AutoGen:
Here are some reasons against choosing AutoGen for an AI Agent project:

1. **Complex Documentation**: The documentation for AutoGen is reported to be difficult to navigate, with insufficient examples, making it challenging for new users to understand and implement the framework effectively.

2. **Limited Built-in Features**: AutoGen may lack certain built-in compliance and features that are available in competing frameworks, potentially limiting its usability in specific applications.

3. **High Cost for Scale**: AutoGen can be expensive to scale up, which may not be sustainable for larger projects or organizations.

4. **Less Intuitive for New Users**: Compared to other frameworks like Crew AI, which uses a more structured and role-based design, AutoGen's dynamic, conversation-driven approach can have a steeper learning curve for beginners.

5. **Limited Ecosystem**: The ecosystem surrounding AutoGen is smaller compared to alternatives like LangChain or OpenAI, which may lead to fewer community resources, plugins, or extensions.

6. **Control and Customization Challenges**: Users may find it difficult to manage certain technical aspects, such as vectorizing content and feeding it back to AI agents, which some other frameworks handle more adeptly.

TERMINATE



## Decision:

Based on the research provided by the team, I recommend using AutoGen for the project. 

The advantages, particularly its efficiency in multi-agent collaboration, ease of use for developers, and robust scalability, outweigh the cons. While the documentation and a steeper learning curve for new users are valid concerns, the benefits of improved collaboration and the backing by Microsoft provide significant long-term value. Furthermore, the open-source nature allows customization which can potentially alleviate some of the documented limitations.

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()